# Lesson 4, T5 twist: deploying speed and clearance

**Complete notebook.** Run the cells in order, top to bottom.

## What this notebook builds

The T5 tracker from Lesson 5 Lab B, wrapped as a service. A stranger uploads a short
video from a fixed camera, says where the two marks are and how far apart they are, and
gets back structured data: each walker's speed, who is slower than the threshold, and the
clearance time at the slowest speed.

The shape is the same as Lesson 4: load the model once, refuse bad input clearly, answer in
a shape a caller can parse. Two things are new because the input is a video and the
answer needs memory across frames:

- A tracker **remembers**. Reusing it between two callers would hand one person's
  identities to the next, so every request gets a fresh tracker and requests are run one
  at a time
- A video can be slow to process, so the limits include **length**, not only size

## The order of work

1. Setup and the weights
2. Load the model once
3. The functions that are the whole service
4. The interface, and a public link
5. Testing the error conditions
6. Cold and warm response time
7. Cloud or edge

---

## 1. Setup

Gradio builds the page. Ultralytics is the same pinned version as Lessons 3 and 5B.
`lap` is the matching library the tracker uses.

In [1]:
ULTRALYTICS_VERSION = "8.4.120"

try:
    import google.colab  # noqa: F401
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    import subprocess
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio", "lap"],
        check=True,
    )

print("Running on Colab:", RUNNING_ON_COLAB)

Running on Colab: True


In [2]:
import json
import math
import os
import subprocess
import sys
import tempfile
import threading
import time
from collections import Counter

import cv2
import gradio as gr
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

print("gradio:", gr.__version__)

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389"
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

WEIGHTS = os.path.join(PROJECT_ROOT, "artifacts", "L3", "best.pt")
print("weights:", WEIGHTS)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
gradio: 6.31.0
Mounted at /content/drive
weights: /content/drive/MyDrive/C389/artifacts/L3/best.pt


### If the weights are missing

`best.pt` is written by the Lesson 3 notebook. If it is not there, the stock model is
used so the lab can still be built and tested. It tracks people well enough and reports
every one of them as `person`, with no direction. Speeds do not need direction, so T5 still
works, and the reply says plainly that the model was not the trained one.

In [3]:
#if os.path.exists(WEIGHTS):
#    MODEL_PATH = WEIGHTS
#    USING_TRAINED_MODEL = True
#else:

MODEL_PATH = "yolo26n.pt"
USING_TRAINED_MODEL = False
print("Using pretrained yolo model")

Using pretrained yolo model


---

## 2. Load the model once

Once, here, at startup, never inside the function that answers a request.

The tracker adds a second rule. Its memory lives **inside** the model object, so the model
is loaded once but the tracker must start empty for every request, or the next caller's
video is matched against the last caller's people. The cell below loads the weights once and
`track_video` in Section 3 resets only the tracker. A lock makes requests take turns,
because two videos tracked at the same moment through one model would mix their identities.

In [4]:
MODEL = YOLO(MODEL_PATH)
CLASS_NAMES = [MODEL.names[key] for key in sorted(MODEL.names)]
PERSON_CLASS_IDS = [index for index, name in enumerate(CLASS_NAMES)
                    if name.startswith("person")]
TRACK_LOCK = threading.Lock()

CONFIDENCE = 0.25            # detections below this are never seen by the tracker
FLICKER_BELOW = 5            # an identity seen in fewer frames than this is ignored
MAX_SECONDS = 90             # longest video accepted
MAX_FRAME_PIXELS = 8_294_400 # 4K, per frame
MIN_MARK_GAP_PCT = 5         # marks closer than this are not two marks
LOW_FPS = 15                 # below this, fast walkers can be missed
SERVICE_VERSION = "1.0.0"

print("model loaded from :", MODEL_PATH)
print("classes           :", CLASS_NAMES)
print("tracked           :", [CLASS_NAMES[index] for index in PERSON_CLASS_IDS])

model loaded from : yolo26n.pt
classes           : ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
tracked           : ['person']


---

## 3. The functions that are the whole service

Validate the video, validate the settings, track, measure, build the reply. Everything
on the page is Gradio, and Gradio is replaceable. These functions would move unchanged into a
container or onto a device.

The marks are taken as a **percentage of the frame width**, not pixels. A caller does not
know the pixel size of their own video, and the same percentage means the same place on a
phone clip and a downscaled one. The reply echoes the pixel positions that were used.

In [5]:
def refusal(status, error, detail):
    return {"status": status, "error": error, "detail": detail}


def validate_video(uploaded):
    """Check that what arrived is a video that can be tracked.

    Args:
        uploaded: a file path, or None when nothing was sent

    Returns:
        (info, error), exactly one of which is None. info carries the path, frame
        rate, frame count, size and length in seconds

    Requirement:
        Refused before any tracking runs: nothing sent, an empty file, a file that
        is not a readable video, a frame rate the file does not state, a video longer
        than MAX_SECONDS, and frames larger than MAX_FRAME_PIXELS. Every refusal says
        what was wrong and what to send instead.
    """
    if uploaded is None or str(uploaded) == "":
        return None, refusal(400, "No video was sent.",
                             "Attach a short MP4 from a fixed camera and send it again.")

    path = str(uploaded)
    if not os.path.isfile(path):
        return None, refusal(400, "No video was sent.", "The file could not be found.")
    if os.path.getsize(path) == 0:
        return None, refusal(400, "The upload was empty.",
                             "The file '%s' is zero bytes." % os.path.basename(path))

    video = cv2.VideoCapture(path)
    try:
        if not video.isOpened():
            return None, refusal(415, "That file is not a readable video.", "Send an MP4.")
        fps = video.get(cv2.CAP_PROP_FPS)
        frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        decoded, _ = video.read()
    finally:
        video.release()

    if not decoded or frames <= 0:
        return None, refusal(415, "That file is not a readable video.",
                             "It opened but no frame could be read. Send an MP4.")
    if not (fps > 0) or math.isnan(fps):
        return None, refusal(422, "The video does not state a frame rate.",
                             "Speed needs the frame rate. Re-record or re-export the clip.")

    seconds = frames / fps
    if seconds > MAX_SECONDS:
        return None, refusal(413, "The video is too long to process.",
                             "%.0f s is beyond the limit of %d s. Send a shorter clip."
                             % (seconds, MAX_SECONDS))
    if width * height > MAX_FRAME_PIXELS:
        return None, refusal(413, "The video frames are too large to process.",
                             "%d by %d pixels is beyond the limit of %d pixels per frame."
                             % (width, height, MAX_FRAME_PIXELS))

    return {"path": path, "fps": fps, "frames": frames, "width": width,
            "height": height, "seconds": seconds}, None


def validate_settings(mark_a_pct, mark_b_pct, distance_m, width_m, slow_ms):
    """Check the numbers the caller supplied. Returns an error dict, or None if fine."""
    named = {"mark A": mark_a_pct, "mark B": mark_b_pct, "distance between marks": distance_m,
             "walkway width": width_m, "slow threshold": slow_ms}
    for name, value in named.items():
        if (value is None or isinstance(value, bool)
                or not isinstance(value, (int, float)) or math.isnan(value)):
            return refusal(422, "The %s is missing or not a number." % name,
                           "Send a number for every setting.")

    if not (0 < mark_a_pct < 100 and 0 < mark_b_pct < 100):
        return refusal(422, "A mark is outside the picture.",
                       "Marks are a percentage of the frame width, between 0 and 100.")
    if abs(mark_a_pct - mark_b_pct) < MIN_MARK_GAP_PCT:
        return refusal(422, "The two marks are too close together.",
                       "Put them at least %d%% of the frame width apart." % MIN_MARK_GAP_PCT)
    if distance_m <= 0 or width_m <= 0:
        return refusal(422, "A distance must be more than zero.",
                       "The distance between the marks and the walkway width are in metres.")
    if slow_ms <= 0:
        return refusal(422, "The slow threshold does not make sense.",
                       "The slow threshold is a speed in m/s and must be above zero.")
    return None

In [6]:
def majority_class(observations):
    return Counter(o["class_name"] for o in observations).most_common(1)[0][0]


def track_video(path, fps, confidence=CONFIDENCE):
    """Track every person in a video. The same output as Lab B's track_clip.

    Returns (tracks, frames_read). tracks maps a track id to observations in time
    order, each with the frame, time in seconds, box, feet point, class and confidence.

    The weights stay loaded. Only the tracker is started fresh, by dropping the
    model's predictor, and the lock makes requests take turns.
    """
    tracks = {}
    index = -1
    with TRACK_LOCK:
        MODEL.predictor = None
        video = cv2.VideoCapture(path)
        try:
            while True:
                ok, frame = video.read()
                if not ok:
                    break
                index = index + 1
                result = MODEL.track(frame, persist=True, tracker="bytetrack.yaml",
                                     conf=confidence, classes=PERSON_CLASS_IDS,
                                     verbose=False)[0]
                if result.boxes is not None and result.boxes.id is not None:
                    for box, track_id, class_id, score in zip(
                            result.boxes.xyxy.tolist(), result.boxes.id.int().tolist(),
                            result.boxes.cls.int().tolist(), result.boxes.conf.tolist()):
                        left, top, right, bottom = box
                        tracks.setdefault(track_id, []).append({
                            "frame": index,
                            "seconds": round(index / fps, 3),
                            "box": [round(v, 1) for v in box],
                            "foot": [round((left + right) / 2, 1), round(bottom, 1)],
                            "class_name": CLASS_NAMES[class_id],
                            "confidence": round(score, 3),
                        })
        finally:
            video.release()
            MODEL.predictor = None
    return tracks, index + 1

In [7]:
def mark_crossing(observations, mark_x):
    """First time the feet pass a vertical mark, as (seconds, direction), or None.

    The time is worked out between the two frames either side of the mark.
    """
    for before, after in zip(observations, observations[1:]):
        x0, x1 = before["foot"][0], after["foot"][0]
        went_right = x0 < mark_x <= x1
        went_left = x0 >= mark_x > x1
        if not (went_right or went_left):
            continue
        fraction = (mark_x - x0) / (x1 - x0)
        seconds = before["seconds"] + fraction * (after["seconds"] - before["seconds"])
        return seconds, ("left to right" if went_right else "right to left")
    return None


def walking_speeds(tracks, mark_a, mark_b, distance_m, slow_ms):
    """Speed of every identity that crossed both marks, and who was left out and why."""
    rows, left_out = [], {}
    for track_id in sorted(tracks):
        observations = tracks[track_id]
        if len(observations) < FLICKER_BELOW:
            left_out[track_id] = "flicker (%d frames)" % len(observations)
            continue
        first = mark_crossing(observations, mark_a)
        second = mark_crossing(observations, mark_b)
        if first is None and second is None:
            continue
        if first is None or second is None:
            left_out[track_id] = "crossed only mark %s" % ("A" if second is None else "B")
            continue
        if first[1] != second[1]:
            left_out[track_id] = "crossed the marks in opposite directions"
            continue
        elapsed = abs(second[0] - first[0])
        if elapsed <= 0:
            left_out[track_id] = "no time between the crossings"
            continue
        speed = distance_m / elapsed
        status = "slow" if speed < slow_ms else "ok"
        rows.append({"track_id": track_id, "class_name": majority_class(observations),
                     "direction": first[1], "seconds_at_a": round(first[0], 3),
                     "seconds_at_b": round(second[0], 3),
                     "seconds_between": round(elapsed, 3),
                     "speed_ms": round(speed, 3), "status": status})
    return rows, left_out


def summarise(rows, width_m):
    """Average, slowest, flagged people and clearance time at the slowest speed.

    With nobody measured the figures are None, which is a valid answer.
    """
    summary = {"walkers": len(rows), "average_ms": None, "slowest_ms": None,
               "slowest_track_id": None, "clearance_seconds_at_slowest": None,
               "flagged_track_ids": []}
    if rows:
        slowest = min(rows, key=lambda r: r["speed_ms"])
        summary["average_ms"] = round(sum(r["speed_ms"] for r in rows) / len(rows), 3)
        summary["slowest_ms"] = slowest["speed_ms"]
        summary["slowest_track_id"] = slowest["track_id"]
        summary["clearance_seconds_at_slowest"] = round(width_m / slowest["speed_ms"], 2)
        summary["flagged_track_ids"] = [r["track_id"] for r in rows if r["status"] == "slow"]
    return summary


In [8]:
def build_reply(uploaded, mark_a_pct=30, mark_b_pct=70, distance_m=4.0, width_m=6.0,
                slow_ms=1.0):
    """The whole service, in one function. Validate, track, measure, answer.

    Returns (reply, picture). A reply carrying "error" failed, and one carrying
    "summary" did not. A video with nobody crossing the marks is a valid answer with
    zero walkers, not an error.
    """
    info, error = validate_video(uploaded)
    if error is not None:
        return error, None
    error = validate_settings(mark_a_pct, mark_b_pct, distance_m, width_m, slow_ms)
    if error is not None:
        return error, None

    mark_a = int(round(info["width"] * mark_a_pct / 100.0))
    mark_b = int(round(info["width"] * mark_b_pct / 100.0))

    started = time.time()
    tracks, frames_read = track_video(info["path"], info["fps"])
    rows, left_out = walking_speeds(tracks, mark_a, mark_b, distance_m, slow_ms)
    summary = summarise(rows, width_m)
    elapsed = time.time() - started

    warnings = []
    if not USING_TRAINED_MODEL:
        warnings.append("The stock model was used, not the trained one. Direction is not reported.")
    if info["fps"] < LOW_FPS:
        warnings.append("Frame rate is %.1f. Fast walkers can be missed below %d." % (info["fps"], LOW_FPS))
    if not tracks:
        warnings.append("Nobody was tracked in this video.")
    elif not rows:
        warnings.append("Nobody crossed both marks. Check the marks are where people walk.")
    if left_out:
        warnings.append("%d identities were left out of the speeds. See left_out." % len(left_out))
    if summary["walkers"] == 1:
        warnings.append("The slowest speed rests on a single walker.")

    reply = {
        "status": 200,
        "summary": summary,
        "people": rows,
        "left_out": {str(k): v for k, v in left_out.items()},
        "warnings": warnings,
        "settings": {"mark_a_px": mark_a, "mark_b_px": mark_b, "mark_distance_m": distance_m,
                     "walkway_width_m": width_m, "slow_threshold_ms": slow_ms},
        "video": {"fps": round(info["fps"], 2), "frames": info["frames"],
                  "seconds": round(info["seconds"], 2),
                  "size": [info["width"], info["height"]]},
        "model_version": SERVICE_VERSION,
        "trained_model": USING_TRAINED_MODEL,
        "processing_seconds": round(elapsed, 3),
    }

    video = cv2.VideoCapture(info["path"])
    ok, first_frame = video.read()
    video.release()
    picture = annotate(first_frame, tracks, rows, mark_a, mark_b) if ok else None

    # Printed rather than kept, so the Colab output below the cell is this service's log.
    print("tracked %d frames in %.2fs: %d identities, %d walkers, clearance %s s"
          % (frames_read, elapsed, len(tracks), summary["walkers"],
             summary["clearance_seconds_at_slowest"]))
    return reply, picture

---

## 4. The interface

The picture shows each walker's path on the first frame, coloured by status (green
fine, red slow, grey not measured), with the two marks drawn on. It is how a
caller checks by eye that the marks landed on the ground marks.

`share=True` is what produces the public link.

In [9]:
def annotate(frame_bgr, tracks, rows, mark_a, mark_b):
    """Paths and marks on the first frame, so the reply can be checked by eye."""
    picture = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    canvas = Image.blend(picture, Image.new("RGB", picture.size, "white"), 0.5)
    draw = ImageDraw.Draw(canvas)
    by_id = {row["track_id"]: row for row in rows}
    colours = {"ok": (72, 160, 88), "slow": (214, 69, 65)}

    for track_id, observations in tracks.items():
        if len(observations) < FLICKER_BELOW:
            continue
        points = [tuple(o["foot"]) for o in observations]
        row = by_id.get(track_id)
        colour = colours[row["status"]] if row else (150, 150, 150)
        if len(points) > 1:
            draw.line(points, fill=colour, width=3)
        label = ("%d: %.2f m/s" % (track_id, row["speed_ms"])) if row else str(track_id)
        draw.text((points[0][0] + 6, points[0][1] - 14), label, fill=colour)

    for name, x in (("A", mark_a), ("B", mark_b)):
        draw.line([(x, 0), (x, canvas.height)], fill=(30, 30, 30), width=3)
        draw.text((x + 6, 6), "mark " + name, fill=(30, 30, 30))
    return canvas


def predict(uploaded, mark_a_pct, mark_b_pct, distance_m, width_m, slow_ms):
    """What the page calls. Returns the annotated picture and the reply."""
    reply, picture = build_reply(uploaded, mark_a_pct, mark_b_pct, distance_m, width_m,
                                 slow_ms)
    return picture, reply

In [10]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Video(label="Walkway video (fixed camera, 90 s at most)"),
        gr.Slider(1, 99, value=30, step=1, label="Mark A, % across the frame"),
        gr.Slider(1, 99, value=70, step=1, label="Mark B, % across the frame"),
        gr.Number(value=4.0, label="Distance between the marks (m), tape measured"),
        gr.Number(value=6.0, label="Full walkway width to cross (m)"),
        gr.Slider(0.2, 2.0, value=1.0, step=0.05, label="Slow walker threshold (m/s)"),
    ],
    outputs=[
        gr.Image(label="Paths and marks"),
        gr.JSON(label="Reply"),
    ],
    title="Walking Speed and Clearance Assistant",
    description="Upload one video from a fixed camera. The reply gives each walker's "
                "speed between the two marks, flags slow walkers, and gives the time "
                "the slowest walker needs to cross the full width.",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee3dd931598d96381f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Two things worth knowing about that link before it goes in a report.

- It lives for 72 hours and dies the moment this runtime stops. It is a demo link, not a deployment
- Anyone holding it can send videos to it. There is no account and no key

Send the link to another team and have them upload a clip from a different phone, with their own
marks and their own tape measurement. A service only ever called by the person who built it has not
been tested.

---

## 5. Testing the error conditions

Seven things a caller will do. Six are not a valid request, and the seventh is a valid video with
nobody in it. What is checked is whether what comes back tells the caller what to do next.

These call `build_reply` directly, not through the page. Gradio's video widget refuses some of
these before the function sees them, and a widget defending a function is not the same as a
function that defends itself.

In [ ]:
scratch = tempfile.mkdtemp()


def write_video(name, frames, fps, size, shade=(200, 200, 200)):
    path = os.path.join(scratch, name)
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
    frame = np.full((size[1], size[0], 3), shade, dtype=np.uint8)
    for _ in range(frames):
        writer.write(frame)
    writer.release()
    return path


text_file = os.path.join(scratch, "not_a_video.mp4")
with open(text_file, "w", encoding="utf-8") as handle:
    handle.write("this is not a video")

empty_file = os.path.join(scratch, "empty.mp4")
open(empty_file, "wb").close()

too_long = write_video("too_long.mp4", frames=MAX_SECONDS + 10, fps=1, size=(64, 48))
too_big = write_video("too_big.mp4", frames=2, fps=30, size=(4200, 2400))
nobody = write_video("nobody.mp4", frames=30, fps=30, size=(640, 480), shade=(214, 210, 202))

CASES = [
    ("nothing sent", (None,), {}),
    ("a text file as an MP4", (text_file,), {}),
    ("an empty file", (empty_file,), {}),
    ("a video that is too long", (too_long,), {}),
    ("frames that are too large", (too_big,), {}),
    ("both marks in the same place", (nobody,), {"mark_a_pct": 50, "mark_b_pct": 50}),
    ("a video with nobody in it", (nobody,), {}),
]

# Collected first, printed afterwards, so the service's own log lines do not run
# through the middle of the table.
rows = []
for label, args, options in CASES:
    reply, _ = build_reply(*args, **options)
    if "error" in reply:
        summary = reply["error"]
    else:
        summary = "walkers %d, warnings %s" % (reply["summary"]["walkers"], reply["warnings"])
    rows.append((label, reply["status"], summary))

print("")
print("%-30s %-7s %s" % ("case", "status", "what came back"))
for label, status, summary in rows:
    print("%-30s %-7s %s" % (label, status, summary))

Copy the status and message for every row into your sheet as they came back, not as expected. If a
row does not say what to do next, fix the message in Section 3.

The last row matters most. A video with nobody in it is **not** a failure. Zero walkers is a correct
answer, and a caller that gets an error for it cannot tell an empty walkway from a broken service.

---

## 6. Cold and warm response time

A cold start is what the first caller waits through after the service has been idle or newly
deployed: importing the libraries, setting up the GPU, reading the weights from a disk that has not
read them. It cannot be measured from inside this notebook, which paid all of that in Section 2.

Video adds a second number the image service did not have. Warm time is not constant, it grows
with the length of the clip, so it is reported as seconds of processing per second of video.

In [ ]:
# Warm, measured here. The clip is the 30 frame blank one from Section 5.
warm = []
for attempt in range(3):
    started = time.time()
    track_video(nobody, 30.0)
    warm.append(time.time() - started)

warm_average = sum(warm) / len(warm)
CLIP_SECONDS = 30 / 30.0
print("warm, 30 frame clip, average : %.3f s" % warm_average)
print("per frame                    : %.1f ms" % (1000 * warm_average / 30))
print("processing per second of video: %.2f s" % (warm_average / CLIP_SECONDS))
print("a %d s clip would take about : %.0f s" % (MAX_SECONDS, MAX_SECONDS * warm_average / CLIP_SECONDS))

In [ ]:
# A separate process, so nothing this notebook has already done counts.
weights = os.path.abspath(MODEL_PATH) if os.path.exists(MODEL_PATH) else MODEL_PATH
probe_path = os.path.join(scratch, "cold_start_probe.py")

with open(probe_path, "w", encoding="utf-8") as handle:
    print("import sys, time", file=handle)
    print("launch = float(sys.argv[1])", file=handle)
    print("start = time.time()", file=handle)
    print("import cv2", file=handle)
    print("from ultralytics import YOLO", file=handle)
    print("imported = time.time()", file=handle)
    print("model = YOLO(%r)" % weights, file=handle)
    print("loaded = time.time()", file=handle)
    print("ok, frame = cv2.VideoCapture(%r).read()" % nobody, file=handle)
    print("model.track(frame, persist=True, tracker='bytetrack.yaml', verbose=False)", file=handle)
    print("answered = time.time()", file=handle)
    print("print('  importing the libraries : %.3f s' % (imported - start))", file=handle)
    print("print('  reading the weights     : %.3f s' % (loaded - imported))", file=handle)
    print("print('  tracking the first frame: %.3f s' % (answered - loaded))", file=handle)
    print("print('ANSWERED %.3f' % (answered - launch))", file=handle)

started = time.time()
probe = subprocess.run([sys.executable, probe_path, repr(started)],
                       capture_output=True, text=True)
process_lifetime = time.time() - started

cold_total = process_lifetime
report = []
for line in probe.stdout.splitlines():
    if not line.strip():
        continue
    if line.startswith("ANSWERED "):
        cold_total = float(line.split()[1])
    else:
        report.append(line)

if probe.returncode != 0:
    print("The probe did not run. Its last words:")
    print(probe.stderr.strip()[-600:])
else:
    print("inside a process that has just started:")
    for line in report:
        print(line)
    print("")
    print("first caller at a cold service waits before the first frame : %.3f s" % cold_total)
    print("warm service, whole 30 frame clip                           : %.3f s" % warm_average)

Read where the time went. For the image service the cold start dwarfed the answer. Here the
warm answer is no longer milliseconds, it scales with the clip, so the two costs are closer and
the question of which one to pay is a real one. Write both down as measured.

---

## 7. Cloud or edge

The same functions could run on a small device at the walkway instead of somewhere else. The trade
is not about which is more modern.

- **Sending a video away** costs far more than sending a photograph: tens of megabytes
  and a connection that holds for the whole upload. It also sends footage of identifiable people
  off the site
- **Running on the device** keeps the footage at the site, and gives up the GPU. Tracking every
  frame of a clip is much heavier than one photograph, so a small device may fall behind real time

Neither is correct in general. A defensible answer names the constraint that decides it for this
walkway: the connection, the cost, how fast the clip has to be processed, or who may see the footage.

### Before leaving

- The public link was sent to another team, and a clip from another phone came back with a sensible answer
- All seven rows of the error table are recorded with the status and message that came back
- Cold and warm timings are written down as measured, including seconds of processing per second of video
- The cloud against edge answer names the constraint that decides it
- `demo.close()` is run below when the lab finishes, or the runtime keeps serving

In [ ]:
demo.close()
print("The share link is closed.")